# Session 6: Agentic RAG Evaluation with Ragas

This notebook evaluates two connected application shapes with Ragas. Breakout Room #1 generates and reviews a small synthetic test set, builds a LangGraph RAG application over a wellness corpus, and compares retrieval strategies. Breakout Room #2 continues with a tool-using metal-price agent and evaluates its trace.

All model requests—including the agent and the LLM-based judges—are routed through **Vercel AI Gateway**. Metals.dev is used only for live price data.

~~~text
wellness corpus -> synthetic Ragas examples -> baseline and candidate RAG scores

live-price request -> LangGraph agent -> tool trace -> agent metrics
~~~

> This is an educational engineering exercise. The wellness corpus is not medical advice, and live metal prices are not investment advice. Verify consequential health and financial information independently.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Generate and review synthetic RAG evaluation examples from a source corpus.
- Build, score, and compare a baseline and candidate LangGraph RAG application.
- Build and inspect a minimal LangGraph ReAct loop.
- Route LangChain and Ragas model calls through Vercel AI Gateway.
- Convert a LangGraph execution history into stable Ragas messages.
- Distinguish tool-call accuracy, agent-goal accuracy, and topic adherence.
- Turn an observed scope failure into a small regression test and guardrail.

## Table of Contents

- **Breakout Room #1: Synthetic RAG Evaluation**
  - Task 1: Environment Setup
  - Task 2: Configure Vercel AI Gateway and Model Roles
  - Task 3: Load the Wellness Corpus
  - Task 4: Generate and Review Synthetic Test Data with Ragas
  - Task 5: Construct a Baseline LangGraph RAG Application
  - Task 6: Evaluate the Baseline with Ragas
  - Task 7: Change One Retrieval Variable and Re-Evaluate
  - Activity #1: Try a Different Retrieval Strategy
- **Breakout Room #2: Agent Evaluation with Ragas**
  - Task 8: Build a ReAct Agent with a Metal-Price Tool
  - Task 9: Implement and Inspect the Agent Graph
  - Task 10: Convert Agent Messages to Ragas Format
  - Task 11: Evaluate Agent Performance with Ragas Metrics
  - Activity #2: Add a Tool-Call Regression Case
  - Activity #3: Design a Scope-Safety Regression
  - Advanced Build: Make Evaluation a Repeatable Regression Suite

---
# Breakout Room #1
## Synthetic RAG Evaluation

This first half restores the RAG-evaluation workflow that precedes the agent-evaluation continuation. We will generate a small reviewable test set from a source corpus, establish a RAG baseline, change one retrieval variable, and use the resulting scores to guide trace inspection.

## Task 1: Environment Setup

From the <code>06_Agentic_RAG_Evaluation</code> folder, create the notebook environment:

~~~bash
uv sync
~~~

Then select the uv-created Python environment as this notebook's kernel.

In [ ]:
from __future__ import annotations

import asyncio
import json
import os
from concurrent.futures import ThreadPoolExecutor
from getpass import getpass
from pathlib import Path
from typing import Annotated, Any
from uuid import uuid4

import instructor
import pandas as pd
import requests
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document
from langchain_core.messages import (
    AIMessage as LCAIMessage,
    HumanMessage as LCHumanMessage,
    SystemMessage as LCSystemMessage,
    ToolMessage as LCToolMessage,
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from openai import OpenAI
from ragas.embeddings.base import embedding_factory
from ragas.llms import llm_factory
from ragas.messages import (
    AIMessage as RagasAIMessage,
    HumanMessage as RagasHumanMessage,
    ToolCall as RagasToolCall,
    ToolMessage as RagasToolMessage,
)
from ragas.metrics.collections import (
    AgentGoalAccuracyWithReference,
    AnswerAccuracy,
    ContextEntityRecall,
    ContextRecall,
    Faithfulness,
    NoiseSensitivity,
    ToolCallAccuracy,
    TopicAdherence,
)
from ragas.run_config import RunConfig
from ragas.testset import TestsetGenerator
from ragas.testset.transforms import (
    CustomNodeFilter,
    default_transforms_for_prechunked,
)
from typing_extensions import TypedDict

## Task 2: Configure Vercel AI Gateway and Model Roles

Vercel AI Gateway provides an OpenAI-compatible endpoint. That means the familiar OpenAI SDK and <code>ChatOpenAI</code> class only need three changes: a Gateway credential, the Gateway base URL, and a provider-qualified model ID such as <code>openai/gpt-5.4-mini</code>.

The notebook uses four model roles:

- **Generator model:** creates synthetic RAG evaluation examples.
- **RAG model:** generates source-grounded answers for the wellness corpus.
- **Judge model:** supplies structured LLM calls for Ragas RAG and agent metrics.
- **Agent model:** decides whether to call the live-price tool and writes the final answer in Breakout Room #2.

The Gateway key can come from <code>AI_GATEWAY_API_KEY</code> for local development or <code>VERCEL_OIDC_TOKEN</code> inside a configured Vercel deployment. Breakout Room #2 separately prompts for <code>METALS_API_KEY</code> when it reaches the live-price tool.

See the [AI Gateway OpenAI-compatible API](https://vercel.com/docs/ai-gateway/openai-compat) and [Python guide](https://vercel.com/docs/ai-gateway/python) for current endpoint and authentication details.

In [ ]:
load_dotenv()

SESSION6_RUNTIME_REVISION = "ragas-sync-adapter-v4"


def read_required_secret(names: tuple[str, ...], prompt: str) -> str:
    for name in names:
        if value := os.environ.get(name):
            return value

    value = getpass(prompt)
    os.environ[names[0]] = value
    return value


gateway_api_key = read_required_secret(
    ("AI_GATEWAY_API_KEY", "VERCEL_OIDC_TOKEN"),
    "Vercel AI Gateway API key: ",
)

GATEWAY_BASE_URL = os.environ.get(
    "AIM_GATEWAY_BASE_URL",
    "https://ai-gateway.vercel.sh/v1",
)
GENERATOR_MODEL_NAME = os.environ.get(
    "AIM_GENERATOR_MODEL",
    "openai/gpt-5.4-mini",
)
RAG_MODEL_NAME = os.environ.get(
    "AIM_RAG_MODEL",
    "openai/gpt-5.4-mini",
)
JUDGE_MODEL_NAME = os.environ.get(
    "AIM_JUDGE_MODEL",
    "openai/gpt-5.4-mini",
)
AGENT_MODEL_NAME = os.environ.get(
    "AIM_AGENT_MODEL",
    "openai/gpt-5.4-mini",
)
EMBEDDING_MODEL_NAME = os.environ.get(
    "AIM_EMBEDDING_MODEL",
    "openai/text-embedding-3-small",
)
TESTSET_SIZE = int(os.environ.get("AIM_TESTSET_SIZE", "4"))
EVAL_CASE_LIMIT = int(os.environ.get("AIM_RAG_EVAL_LIMIT", "3"))
MAX_CONCURRENCY = int(os.environ.get("AIM_MAX_CONCURRENCY", "2"))

for role, model_name in {
    "generator": GENERATOR_MODEL_NAME,
    "rag": RAG_MODEL_NAME,
    "judge": JUDGE_MODEL_NAME,
    "agent": AGENT_MODEL_NAME,
    "embedding": EMBEDDING_MODEL_NAME,
}.items():
    if "/" not in model_name:
        raise ValueError(
            f"{role} model must use a provider-qualified AI Gateway ID; got "
            f"{model_name!r}."
        )

gateway_sync_client = OpenAI(
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

# Ragas generation needs structured output for graph transforms and sample creation.
generator_llm = llm_factory(
    GENERATOR_MODEL_NAME,
    provider="openai",
    client=gateway_sync_client,
    mode=instructor.Mode.TOOLS,
    max_tokens=2048,
)
generator_llm.model_args = {"max_tokens": 2048, "max_retries": 3}
generator_embeddings = embedding_factory(
    "openai",
    model=EMBEDDING_MODEL_NAME,
    client=gateway_sync_client,
)

rag_embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
    check_embedding_ctx_length=False,
)
rag_llm = ChatOpenAI(
    model=RAG_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
    max_retries=2,
    timeout=60,
)
agent_llm = ChatOpenAI(
    model=AGENT_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
    max_retries=2,
    timeout=60,
)

if generator_llm.is_async:
    raise RuntimeError(
        "Session 6 requires a synchronous Ragas generation client. "
        "Reload the notebook and rerun the environment/configuration cells."
    )


# Ragas metric methods call agenerate(), while Instructor's AsyncOpenAI
# path is incompatible with this Jupyter/Python runtime. Keep every actual
# Gateway request synchronous, then bridge only the Ragas coroutine boundary.
def build_sync_judge_llm():
    # Construct inside each scoring worker so a notebook cannot reuse a
    # previous kernel's async client by accident.
    judge = llm_factory(
        JUDGE_MODEL_NAME,
        provider="openai",
        client=OpenAI(api_key=gateway_api_key, base_url=GATEWAY_BASE_URL),
        mode=instructor.Mode.TOOLS,
        max_tokens=1024,
    )
    judge.model_args = {"max_tokens": 1024, "max_retries": 3}

    async def agenerate_from_sync(prompt, response_model):
        return await asyncio.to_thread(
            judge.generate,
            prompt=prompt,
            response_model=response_model,
        )

    judge.agenerate = agenerate_from_sync
    return judge


judge_llm = build_sync_judge_llm()
if judge_llm.is_async:
    raise RuntimeError("Session 6 requires a synchronous Ragas judge client.")

# Repair a previously executed Task 6 cell when this configuration cell is
# rerun in an existing notebook kernel.
if "rag_metrics" in globals():
    rag_metrics = {
        "context_recall": ContextRecall(llm=judge_llm),
        "faithfulness": Faithfulness(llm=judge_llm),
        "answer_accuracy": AnswerAccuracy(llm=judge_llm),
        "context_entity_recall": ContextEntityRecall(llm=judge_llm),
        "noise_sensitivity": NoiseSensitivity(llm=judge_llm, mode="relevant"),
    }


ragas_run_config = RunConfig(
    timeout=180,
    max_retries=3,
    max_wait=30,
    max_workers=MAX_CONCURRENCY,
)

# Jupyter owns an event loop. Run Ragas coroutines in a dedicated worker;
# their model requests use the synchronous adapter above.
def run_ragas_sync(func, *args, **kwargs):
    with ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(func, *args, **kwargs).result()


def run_ragas_async(async_function, *args, **kwargs):
    # Accept both the current function-plus-arguments form and the older
    # pre-v4 coroutine form so a partially rerun notebook can recover.
    if asyncio.iscoroutine(async_function):
        return run_ragas_sync(asyncio.run, async_function)

    def invoke():
        return asyncio.run(async_function(*args, **kwargs))

    return run_ragas_sync(invoke)


print(f"AI Gateway: {GATEWAY_BASE_URL}")
print(f"Generator model: {GENERATOR_MODEL_NAME}")
print(f"RAG model: {RAG_MODEL_NAME}")
print(f"Judge model: {JUDGE_MODEL_NAME}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Session 6 runtime revision: {SESSION6_RUNTIME_REVISION}")
print("Ragas judge: synchronous Gateway client with async-safe adapter")
print(f"Synthetic examples: {TESTSET_SIZE}; evaluated examples: {EVAL_CASE_LIMIT}")

## Task 3: Load the Wellness Corpus

Breakout Room #1 restores the RAG-evaluation workflow that comes before the agent-evaluation continuation. We use a small, source-linked wellness corpus so that every generated test question, retrieved context, and answer has a visible provenance.

> This corpus is an educational retrieval artifact, not medical advice. The RAG application must preserve that boundary and say when the context is insufficient.

In [ ]:
corpus_path = Path("data/HealthWellnessGuide.txt")
if not corpus_path.exists():
    raise FileNotFoundError(
        f"Expected the course corpus at {corpus_path.resolve()}"
    )

source_documents = TextLoader(str(corpus_path), encoding="utf-8").load()
generation_splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=100,
)
generation_chunks = generation_splitter.split_documents(source_documents)

print(f"Source documents: {len(source_documents)}")
print(f"Generation chunks: {len(generation_chunks)}")
print(generation_chunks[0].page_content[:900])

## Task 4: Generate and Review Synthetic Test Data with Ragas

Ragas can enrich pre-chunked source material, build relationships between chunks, and synthesize candidate questions, reference contexts, and reference answers. The generated rows are not automatically ground truth: inspect them before treating them as evaluation examples.

The current pre-chunked Ragas workflow is used here instead of the deprecated <code>LangchainLLMWrapper</code> path from the source notebook. Generation, embeddings, and structured outputs all use Vercel AI Gateway.

In [ ]:
# The current Ragas pre-chunked pipeline includes a parent-child filter that
# is not applicable to our independent text chunks, so remove it explicitly.
generation_transforms = [
    transform
    for transform in default_transforms_for_prechunked(
        llm=generator_llm,
        embedding_model=generator_embeddings,
    )
    if not isinstance(transform, CustomNodeFilter)
]

testset_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)
# Ragas' generation transforms make internal async Instructor calls. Keep
# them off the Jupyter event loop for the same reason as the metric calls.
synthetic_testset = run_ragas_sync(
    testset_generator.generate_with_chunks,
    chunks=generation_chunks,
    testset_size=TESTSET_SIZE,
    transforms=generation_transforms,
    run_config=ragas_run_config,
)
synthetic_testset_df = synthetic_testset.to_pandas()

synthetic_testset_df[[
    "user_input",
    "reference",
    "reference_contexts",
]].head()

### Curate Before You Score

Review every candidate row. Keep only questions that are answerable from the supplied reference contexts, whose reference answer is supported by those contexts, and whose wording respects the corpus's safety boundary. The code below limits the worked evaluation to a small reviewable subset.

In [ ]:
required_testset_columns = [
    "user_input",
    "reference",
    "reference_contexts",
]
missing_columns = set(required_testset_columns) - set(synthetic_testset_df.columns)
if missing_columns:
    raise RuntimeError(
        "Ragas did not return the expected evaluation columns: "
        f"{sorted(missing_columns)}"
    )

# In a production workflow, replace this with an explicit reviewed-status filter.
reviewed_testset_df = (
    synthetic_testset_df.loc[:, required_testset_columns]
    .head(EVAL_CASE_LIMIT)
    .copy()
)
reviewed_testset_df

## Task 5: Construct a Baseline LangGraph RAG Application

The baseline uses dense similarity retrieval over an in-memory Qdrant index. Its graph is intentionally simple:

~~~text
question -> retrieve source chunks -> generate from retrieved context
~~~

All embeddings and answer-model calls use AI Gateway. The prompt confines answers to retrieved course context and preserves the wellness safety boundary.

In [ ]:
RAG_SYSTEM_PROMPT = """You are an educational wellness-information assistant.

Answer only from the retrieved course context. If the context does not provide
enough information, say so. Do not diagnose, prescribe, or provide individualized
medical, nutrition, or mental-health advice. Preserve urgent-care and crisis
boundaries that appear in the context.
"""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", "Question:\n{question}\n\nRetrieved context:\n{context}"),
    ]
)


class RAGState(TypedDict):
    question: str
    context: list[Document]
    answer: str


def build_vector_store(documents: list[Document]) -> QdrantVectorStore:
    return QdrantVectorStore.from_documents(
        documents=documents,
        embedding=rag_embeddings,
        location=":memory:",
        collection_name=f"wellness_eval_{uuid4().hex[:8]}",
    )


def build_rag_graph(retriever):
    def retrieve(state: RAGState):
        return {"context": retriever.invoke(state["question"])}

    def generate(state: RAGState):
        context_text = "\n\n".join(
            document.page_content for document in state["context"]
        )
        messages = rag_prompt.format_messages(
            question=state["question"],
            context=context_text,
        )
        response = rag_llm.invoke(messages)
        return {"answer": str(response.content)}

    graph = StateGraph(RAGState)
    graph.add_node("retrieve", retrieve)
    graph.add_node("generate", generate)
    graph.add_edge(START, "retrieve")
    graph.add_edge("retrieve", "generate")
    return graph.compile()

In [ ]:
RAG_CHUNK_SIZE = 500
RAG_CHUNK_OVERLAP = 75
BASELINE_RETRIEVAL_K = 3

rag_splitter = RecursiveCharacterTextSplitter(
    chunk_size=RAG_CHUNK_SIZE,
    chunk_overlap=RAG_CHUNK_OVERLAP,
)
rag_documents = rag_splitter.split_documents(source_documents)
vector_store = build_vector_store(rag_documents)
baseline_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": BASELINE_RETRIEVAL_K},
)
baseline_graph = build_rag_graph(baseline_retriever)

spot_check = baseline_graph.invoke(
    {"question": "Which habits in the guide can support a consistent sleep routine?"}
)
print(spot_check["answer"])
print(f"\nRetrieved chunks: {len(spot_check['context'])}")

#### Question #1

What is the purpose of <code>chunk_overlap</code> in a recursive text splitter? What tradeoff does increasing overlap introduce?

##### Answer

Your answer here.

## Task 6: Evaluate the Baseline with Ragas

We now run the reviewed synthetic questions through the baseline graph and preserve the question, reference answer, retrieved contexts, and generated answer together. The current Ragas collections API scores each row directly, which keeps the inputs visible and routes every judge call through AI Gateway.

The worked set uses five complementary signals: context recall, faithfulness, answer accuracy, context-entity recall, and noise sensitivity. Scores are directional evidence; inspect individual rows before deciding why an average changed.

In [ ]:
def as_context_list(value: Any) -> list[str]:
    if isinstance(value, str):
        return [value]
    return [str(item) for item in value]


def run_rag_over_testset(graph, dataframe: pd.DataFrame) -> list[dict[str, Any]]:
    rows = []
    for _, row in dataframe.iterrows():
        result = graph.invoke({"question": row["user_input"]})
        rows.append(
            {
                "user_input": row["user_input"],
                "reference": row["reference"],
                "reference_contexts": as_context_list(row["reference_contexts"]),
                "retrieved_contexts": [
                    document.page_content for document in result["context"]
                ],
                "response": result["answer"],
            }
        )
    return rows


baseline_rows = run_rag_over_testset(baseline_graph, reviewed_testset_df)
pd.DataFrame(baseline_rows)[["user_input", "response", "reference"]]

In [ ]:
async def score_rag_rows(rows: list[dict[str, Any]]) -> pd.DataFrame:
    judge_llm = build_sync_judge_llm()
    rag_metrics = {
        "context_recall": ContextRecall(llm=judge_llm),
        "faithfulness": Faithfulness(llm=judge_llm),
        "answer_accuracy": AnswerAccuracy(llm=judge_llm),
        "context_entity_recall": ContextEntityRecall(llm=judge_llm),
        "noise_sensitivity": NoiseSensitivity(llm=judge_llm, mode="relevant"),
    }

    score_rows = []
    for index, row in enumerate(rows, start=1):
        score_rows.append(
            {
                "case": index,
                "context_recall": (await rag_metrics["context_recall"].ascore(
                    user_input=row["user_input"],
                    retrieved_contexts=row["retrieved_contexts"],
                    reference=row["reference"],
                )).value,
                "faithfulness": (await rag_metrics["faithfulness"].ascore(
                    user_input=row["user_input"],
                    response=row["response"],
                    retrieved_contexts=row["retrieved_contexts"],
                )).value,
                "answer_accuracy": (await rag_metrics["answer_accuracy"].ascore(
                    user_input=row["user_input"],
                    response=row["response"],
                    reference=row["reference"],
                )).value,
                "context_entity_recall": (await rag_metrics["context_entity_recall"].ascore(
                    reference=row["reference"],
                    retrieved_contexts=row["retrieved_contexts"],
                )).value,
                "noise_sensitivity": (await rag_metrics["noise_sensitivity"].ascore(
                    user_input=row["user_input"],
                    response=row["response"],
                    reference=row["reference"],
                    retrieved_contexts=row["retrieved_contexts"],
                )).value,
            }
        )
    return pd.DataFrame(score_rows)


baseline_scores = run_ragas_async(score_rag_rows, baseline_rows)
baseline_summary = baseline_scores.drop(columns="case").mean().to_frame("baseline")
baseline_summary

## Task 7: Change One Retrieval Variable and Re-Evaluate

The source notebook used Cohere reranking. To keep all model calls on AI Gateway, this update uses maximal marginal relevance (MMR) instead: it retrieves a wider candidate set and balances similarity with diversity. The corpus, embeddings, answer model, prompt, and evaluation set remain fixed.

This is a controlled retrieval experiment, not a claim that MMR is always better. Inspect changes in both the aggregate scores and the individual traces.

In [ ]:
CANDIDATE_RETRIEVAL_K = min(3, len(rag_documents))
CANDIDATE_FETCH_K = min(12, len(rag_documents))
CANDIDATE_LAMBDA_MULT = 0.30

candidate_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": CANDIDATE_RETRIEVAL_K,
        "fetch_k": CANDIDATE_FETCH_K,
        "lambda_mult": CANDIDATE_LAMBDA_MULT,
    },
)
candidate_graph = build_rag_graph(candidate_retriever)
candidate_rows = run_rag_over_testset(candidate_graph, reviewed_testset_df)
candidate_scores = run_ragas_async(score_rag_rows, candidate_rows)

comparison = pd.concat(
    [
        baseline_scores.assign(experiment="baseline_similarity"),
        candidate_scores.assign(experiment="candidate_mmr"),
    ],
    ignore_index=True,
)
comparison.groupby("experiment").mean(numeric_only=True).T

#### Question #2

Which experiment performed better on which metric? Inspect at least one trace before explaining why; do not infer a cause from the aggregate alone.

##### Answer

Your answer here.

#### Question #3

What are the practical strengths and limitations of synthetic test data for RAG evaluation? Include one way a synthetic set can mislead you.

##### Answer

Your answer here.

#### Question #4

For an educational wellness assistant, which of the five worked metrics would you prioritize and why? What additional human review would still be necessary?

##### Answer

Your answer here.

## Activity #1: Try a Different Retrieval Strategy

Create one more controlled experiment. Change a single retrieval variable—such as similarity <code>k</code>, MMR <code>fetch_k</code>, MMR <code>lambda_mult</code>, or chunk overlap—then rebuild only the components that must change.

Requirements:

1. State the one variable you will change and your prediction.
2. Keep the reviewed evaluation rows and answer prompt fixed.
3. Run the new graph and score it with the same metrics.
4. Compare aggregate scores and inspect at least one changed trace.
5. Record a quality, cost, or latency tradeoff.

In [ ]:
# YOUR CODE HERE
#
# student_retriever = vector_store.as_retriever(
#     search_type="mmr",
#     search_kwargs={"k": 4, "fetch_k": 16, "lambda_mult": 0.5},
# )
# student_graph = build_rag_graph(student_retriever)
# student_rows = run_rag_over_testset(student_graph, reviewed_testset_df)
# student_scores = run_ragas_async(score_rag_rows, student_rows)
# student_scores.drop(columns="case").mean()

### Activity #1 Findings

- Variable changed:
- Prediction:
- Baseline result:
- Candidate result:
- Trace inspected:
- Decision:
- Cost or latency tradeoff:

---
# Breakout Room #2
## Agent Evaluation with Ragas

This continuation turns the evaluation contract from Breakout Room #1 into a live LangGraph agent exercise. We will build a ReAct agent, convert its execution history to Ragas messages, and score its process, outcome, and scope behavior.

## Task 8: Build a ReAct Agent with a Live Metal-Price Tool

The tool is deliberately narrow: it returns the live USD price **per gram** for a supported metal. The tool itself owns live-data access and error handling, so the model never sees the API key and never needs to invent a price.

Metals.dev's <code>/v1/latest</code> endpoint accepts an API key, currency, and unit. We request <code>currency=USD</code> and <code>unit=g</code>, then return a compact JSON string that the agent can cite in its response.

In [ ]:
metals_api_key = read_required_secret(
    ("METALS_API_KEY", "METAL_API_KEY"),
    "Metals.dev API key: ",
)

SUPPORTED_METALS = {
    "gold",
    "silver",
    "platinum",
    "palladium",
    "copper",
    "aluminum",
    "nickel",
    "lead",
    "zinc",
}
METAL_ALIASES = {"aluminium": "aluminum"}


@tool
def get_metal_price(metal_name: str) -> str:
    """Return the current USD spot price per gram for one supported metal.

    Use this tool whenever a user asks for a current or live metal price.
    Supported metals are gold, silver, platinum, palladium, copper, aluminum,
    nickel, lead, and zinc. The response is live market data, not investment
    advice.
    """
    metal = METAL_ALIASES.get(metal_name.lower().strip(), metal_name.lower().strip())

    if metal not in SUPPORTED_METALS:
        return json.dumps(
            {
                "error": f"Unsupported metal: {metal_name!r}",
                "supported_metals": sorted(SUPPORTED_METALS),
            }
        )

    try:
        response = requests.get(
            "https://api.metals.dev/v1/latest",
            params={
                "api_key": metals_api_key,
                "currency": "USD",
                "unit": "g",
            },
            headers={"Accept": "application/json"},
            timeout=20,
        )
    except requests.RequestException:
        return json.dumps(
            {"error": "Metals.dev could not be reached. Please try again later."}
        )

    if not response.ok:
        return json.dumps(
            {"error": f"Metals.dev returned HTTP {response.status_code}."}
        )

    try:
        payload = response.json()
    except ValueError:
        return json.dumps({"error": "Metals.dev returned invalid JSON."})

    if payload.get("status") != "success":
        return json.dumps({"error": "Metals.dev did not return a price."})

    price = payload.get("metals", {}).get(metal)
    if not isinstance(price, (int, float)):
        return json.dumps(
            {"error": f"No live price was returned for {metal}."}
        )

    return json.dumps(
        {
            "metal": metal,
            "price_usd_per_gram": float(price),
            "currency": payload.get("currency", "USD"),
            "unit": payload.get("unit", "g"),
            "timestamp": payload.get("timestamp"),
        },
        sort_keys=True,
    )

## Task 9: Implement and Inspect the LangGraph ReAct Loop

The graph has two nodes:

1. **assistant** asks the model for the next action.
2. **tools** executes a requested tool call.

A conditional edge returns to the tool node when the assistant has tool calls; otherwise the graph ends. We compile two variants with the same tool and model:

- A **baseline** agent that is generally helpful.
- A **guarded** agent that must politely refuse requests outside live metal prices.

Keeping everything else fixed lets us later attribute a topic-adherence change to the scope instruction.

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[Any], add_messages]


BASELINE_SYSTEM_PROMPT = """
You are a helpful assistant. When a user asks for a current metal spot price,
call get_metal_price instead of inventing a number. Clearly label live price
information as informational, not financial advice.
""".strip()

GUARDED_SYSTEM_PROMPT = """
You are a narrow live-metal-price assistant. Your only job is to help with
current USD spot prices for supported metals. For a current price request, call
get_metal_price rather than inventing a number. If a request is unrelated to
live metal prices, politely say that you can only help with live metal prices.
Do not provide investment, trading, allocation, or tax advice.
""".strip()

tools = [get_metal_price]


def build_metal_agent(system_prompt: str):
    model_with_tools = agent_llm.bind_tools(tools)

    def assistant(state: AgentState):
        response = model_with_tools.invoke(
            [LCSystemMessage(content=system_prompt), *state["messages"]]
        )
        return {"messages": [response]}

    def should_continue(state: AgentState):
        last_message = state["messages"][-1]
        return "tools" if getattr(last_message, "tool_calls", []) else END

    graph = StateGraph(AgentState)
    graph.add_node("assistant", assistant)
    graph.add_node("tools", ToolNode(tools))
    graph.add_edge(START, "assistant")
    graph.add_conditional_edges("assistant", should_continue, ["tools", END])
    graph.add_edge("tools", "assistant")
    return graph.compile()


baseline_agent = build_metal_agent(BASELINE_SYSTEM_PROMPT)
guarded_agent = build_metal_agent(GUARDED_SYSTEM_PROMPT)

In [ ]:
print(baseline_agent.get_graph().draw_mermaid())

### Run and Inspect One Trace

We begin with a request that should need exactly one tool call. The helper keeps the full message list so we can inspect and score the same evidence.

In [ ]:
def run_turn(agent, user_text: str, history: list[Any] | None = None) -> list[Any]:
    messages = [*(history or []), LCHumanMessage(content=user_text)]
    result = agent.invoke({"messages": messages})
    return result["messages"]


def show_messages(messages: list[Any]) -> None:
    for index, message in enumerate(messages, start=1):
        print(f"\n--- Message {index}: {message.type} ---")
        print(message.pretty_repr())


agent_evaluation_contract = {
    "request": "What is the live USD spot price of copper per gram?",
    "reference_tool_calls": [
        RagasToolCall(name="get_metal_price", args={"metal_name": "copper"})
    ],
    "allowed_topics": ["live metal spot prices"],
}

copper_messages = run_turn(
    baseline_agent,
    agent_evaluation_contract["request"],
)
show_messages(copper_messages)

## Task 10: Normalize a LangGraph Trace for Ragas

Ragas evaluates its own message schema. Instead of depending on provider-specific raw payloads, this adapter uses LangChain's normalized <code>AIMessage.tool_calls</code> field. That makes the evaluation layer more stable when model providers or SDK response shapes change.

System messages guide the run but are intentionally excluded from the trace under evaluation.

In [ ]:
def content_as_text(content: Any) -> str:
    if isinstance(content, str):
        return content
    return json.dumps(content, ensure_ascii=False, default=str)


def to_ragas_messages(messages: list[Any]) -> list[Any]:
    converted = []

    for message in messages:
        if isinstance(message, LCHumanMessage):
            converted.append(RagasHumanMessage(content=content_as_text(message.content)))
        elif isinstance(message, LCAIMessage):
            tool_calls = [
                RagasToolCall(
                    name=tool_call["name"],
                    args=dict(tool_call.get("args") or {}),
                )
                for tool_call in message.tool_calls
            ]
            converted.append(
                RagasAIMessage(
                    content=content_as_text(message.content),
                    tool_calls=tool_calls or None,
                )
            )
        elif isinstance(message, LCToolMessage):
            converted.append(RagasToolMessage(content=content_as_text(message.content)))
        elif isinstance(message, LCSystemMessage):
            continue
        else:
            raise TypeError(f"Unsupported LangChain message: {type(message).__name__}")

    return converted


copper_trace = to_ragas_messages(copper_messages)
for index, message in enumerate(copper_trace, start=1):
    print(f"\n--- Ragas message {index}: {message.type} ---")
    print(message.pretty_repr())

#### Question #5

Why is it useful to score the same normalized trace that you inspect manually, rather than logging one representation and evaluating a different one?

##### Answer

Your answer here.

## Task 11: Evaluate Agent Performance with Ragas Metrics

Different metrics answer different questions. A high final-answer score does not prove that the agent followed the desired process, and a correct tool call does not prove that the application stayed in scope.

### Tool-Call Accuracy

<code>ToolCallAccuracy</code> is a deterministic process metric. It checks the tool-call sequence and arguments against a reference. Here we expect precisely one call to <code>get_metal_price</code> with <code>metal_name="copper"</code>.

The modern Ragas collection API returns a <code>MetricResult</code>; its <code>value</code> is between 0 and 1.

In [ ]:
async def score_tool_call_accuracy():
    return await ToolCallAccuracy(strict_order=True).ascore(
        user_input=copper_trace,
        reference_tool_calls=agent_evaluation_contract["reference_tool_calls"],
    )


tool_call_result = run_ragas_async(score_tool_call_accuracy)

print(f"Tool-call accuracy: {tool_call_result.value:.2f}")

A score below 1 is not automatically a model failure. Inspect the trace first. Common causes include a misspelled metal, a missing tool call, an extra tool call, or an otherwise reasonable choice whose argument does not match the test's expected contract.

### Agent Goal Accuracy

Tool-call accuracy measures the process. <code>AgentGoalAccuracyWithReference</code> asks an LLM judge whether the final workflow outcome meets a stated reference. This is useful when multiple valid tool paths could satisfy the user.

Unlike the previous metric, goal accuracy is LLM-based. It makes structured judge calls through AI Gateway, so treat it as a useful signal to inspect—not a source of unquestionable truth.

In [ ]:
silver_messages = run_turn(
    baseline_agent,
    "What is the live USD spot price of 10 grams of silver?",
)
silver_trace = to_ragas_messages(silver_messages)

async def score_goal_accuracy():
    return await AgentGoalAccuracyWithReference(
        llm=build_sync_judge_llm()
    ).ascore(
        user_input=silver_trace,
        reference=(
            "Report the current USD spot price for 10 grams of silver using the "
            "live tool result, including a clear informational-not-advice boundary."
        ),
    )


goal_result = run_ragas_async(score_goal_accuracy)

show_messages(silver_messages)
print(f"Agent-goal accuracy: {goal_result.value:.2f}")

#### Question #6

Give one example in which tool-call accuracy could be 1.0 while agent-goal accuracy is low. Give another in which goal accuracy could be high while the exact expected tool call was not used.

##### Answer

Your answer here.

### Topic Adherence and a Scope Guardrail

A narrow tool does not, by itself, keep a general-purpose language model from answering unrelated questions. We will compare two otherwise identical agents on a two-turn transcript:

1. An in-scope copper-price request.
2. An unrelated question about eagle flight speed.

The baseline is allowed to be generally helpful; the guarded version should decline the unrelated request. We use **precision** mode because it asks: of the topics the agent actually answered, how many were inside the approved live-metal-price scope?

In [ ]:
def run_scope_case(agent) -> list[Any]:
    history = run_turn(
        agent,
        "What is the live USD spot price of copper per gram?",
    )
    return run_turn(agent, "How fast can an eagle fly?", history=history)


baseline_scope_messages = run_scope_case(baseline_agent)
guarded_scope_messages = run_scope_case(guarded_agent)

baseline_scope_trace = to_ragas_messages(baseline_scope_messages)
guarded_scope_trace = to_ragas_messages(guarded_scope_messages)

async def score_topic_adherence(trace):
    topic_metric = TopicAdherence(
        llm=build_sync_judge_llm(),
        mode="precision",
    )
    return await topic_metric.ascore(
        user_input=trace,
        reference_topics=agent_evaluation_contract["allowed_topics"],
    )


baseline_topic_result = run_ragas_async(
    score_topic_adherence,
    baseline_scope_trace,
)
guarded_topic_result = run_ragas_async(
    score_topic_adherence,
    guarded_scope_trace,
)

print(f"Baseline topic-adherence precision: {baseline_topic_result.value:.2f}")
print(f"Guarded topic-adherence precision: {guarded_topic_result.value:.2f}")

print("\nBaseline trace:")
show_messages(baseline_scope_messages)
print("\nGuarded trace:")
show_messages(guarded_scope_messages)

The comparison is deliberately small, not a production scorecard. If the guarded score does not improve, inspect the messages before changing the metric: perhaps the model answered the unrelated question anyway, the refusal was ambiguous, or the judge classified a topic differently from your product definition.

#### Question #7

Why is a higher topic-adherence score not enough by itself to prove that a production agent is safe or useful? Name at least two kinds of evidence you would add.

##### Answer

Your answer here.

## Activity #2: Add a Tool-Call Regression Case

Create a new test case for a supported metal. Keep the request unambiguous enough that you can state the expected tool call precisely.

Requirements:

1. Choose a new supported metal, such as platinum or palladium.
2. Run the baseline agent and inspect the trace.
3. Convert the trace with <code>to_ragas_messages</code>.
4. Define the matching <code>RagasToolCall</code>.
5. Score it with strict-order <code>ToolCallAccuracy</code>.
6. Record what you would expect to change if the tool schema gained a second required argument.

In [ ]:
# YOUR CODE HERE
#
# regression_messages = run_turn(baseline_agent, "...")
# regression_trace = to_ragas_messages(regression_messages)
# async def score_regression():
#     return await ToolCallAccuracy(strict_order=True).ascore(
#         user_input=regression_trace,
#         reference_tool_calls=[
#             RagasToolCall(name="get_metal_price", args={"metal_name": "..."})
#         ],
#     )
#
# regression_result = run_ragas_async(score_regression)
# print(regression_result.value)

### Activity #2 Notes

- Request:
- Expected tool call:
- Score:
- Trace evidence:
- If the tool schema changed:


## Activity #3: Design a Scope-Safety Regression

Choose an out-of-scope request that a broadly helpful model might answer, then turn it into a comparison between the baseline and guarded agents. Avoid requests for real financial advice; the point is to test the product boundary, not to solicit advice.

Requirements:

1. State the intended product boundary in one sentence.
2. Write an in-scope turn and an out-of-scope turn.
3. Run both agents with the same two-turn transcript.
4. Measure topic-adherence precision with the same approved topic list.
5. Inspect both traces.
6. Decide whether the guardrail improved the behavior for the reason you expected.
7. Note one way that an overly strict guardrail could harm user experience.

In [ ]:
# YOUR CODE HERE
#
# def run_my_scope_case(agent):
#     history = run_turn(agent, "... in-scope request ...")
#     return run_turn(agent, "... out-of-scope request ...", history=history)
#
# baseline_messages = run_my_scope_case(baseline_agent)
# guarded_messages = run_my_scope_case(guarded_agent)
# pass judge_llm into TopicAdherence; its async method is safely bridged
# to the synchronous Gateway client by the configured adapter.

### Activity #3 Notes

- Product boundary:
- In-scope request:
- Out-of-scope request:
- Baseline score and trace evidence:
- Guarded score and trace evidence:
- Did the guardrail help for the expected reason?:
- Potential user-experience cost of the guardrail:


## Advanced Build: Make Evaluation a Repeatable Regression Suite

Move the examples out of notebook cells and into a small versioned dataset, for example JSONL with fields for <code>name</code>, <code>messages</code>, <code>reference_tool_calls</code>, <code>reference_goal</code>, and <code>allowed_topics</code>.

Then write a runner that:

1. Executes each case against a named model and prompt version.
2. Saves the normalized trace and metric values.
3. Fails a CI check only on thresholds you have reviewed deliberately.
4. Reports cost and latency beside quality scores.
5. Keeps a small hand-reviewed set of difficult, realistic failures.

Treat the suite as a living product contract. Add a case whenever a real failure teaches you something, and retire cases only with an explicit reason.

## Final Takeaways

- A trace makes agent behavior inspectable.
- Tool-call accuracy checks an expected process; goal accuracy checks an outcome.
- Topic adherence reveals whether a product boundary is actually reflected in behavior.
- A metric becomes useful when it is paired with trace inspection, a concrete hypothesis, and a controlled change.
- AI Gateway lets the agent and the judges share one provider-agnostic integration point.